In [36]:
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage
from langchain_openai.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START,END ,MessagesState
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langgraph.store.base import BaseStore


load_dotenv()
llm =  ChatOpenAI(model="gpt-4o-mini")

## Table of contents 
- Creating existing memories of the user 
- Generating a system prompt. 
- Passing the system prompt to the user and creating the note for it 

**Note: This notebook includes the basic implementation of the chatbot using the long-term memory, but it is highly recommended to watch the long-term memory full video. And the code**

**Creating existing memories of the user**

In [37]:
store = InMemoryStore()

# creating a namespace
namespace = ("user", "u1","details")
# Adding memories
store.put(namespace, "1", {"data": "Hi, my name is Rishabh. "})
store.put(namespace, "2", {"data": "I am a programming enthusiast in machine learning and deep learning and love Python as a language. "})
store.put(namespace, "3",{"data":"My background is civil engineering. "})


**Writing a system prompt to the AI to reply in a personalized form**

In [38]:
SYSTEM_PROMPT =  """

You are a personal assistant chatbot, and your job is to personalize the responses based on the personal data of the user. 
Always remember these few points when answering the user query. 
1. Use the name as much as possible.
2. Relate to the user's interest and inclination towards the subject and projects.
3. Try to ask queries based on the user's interest and responses.


USER DATA : {memories}

"""

In [60]:
def chat_node(state : MessagesState , config :  RunnableConfig , store : BaseStore  ) :
    
    # retriveing_all_memeories 
    user_id =  config["configurable"]["user_id"]
    user_details = ("user", user_id, "details")
    items = store.search(user_details)
    
    all_values  =  "".join([i.value["data"] for i in items])
    
    # passing it to system prompt and then prompt to AI 
    system_prompt =  SYSTEM_PROMPT.format(memories=all_values)
    
    ans =  llm.invoke([SystemMessage(content=system_prompt)] +  state["messages"])
    
    return {"messages":ans}
    
    

In [49]:
graph = StateGraph(MessagesState)
graph.add_node("chat_node",chat_node)



graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)


checkpointer =  InMemorySaver()
workflow =  graph.compile(checkpointer=checkpointer,store=store)


In [59]:
config1 =  {"configurable":{"user_id":"u1","thread_id":"chat1"}}
result = workflow.invoke({"messages":[
    {"role":"user","content":"Give me programming project ideas. "}
]},config=config1)

print(result['messages'][1].content )

Hi, my name is Rishabh. I am a programming enthusiast in machine learning and deep learning and love Python as a language. My background is civil engineering. 
Sure, Rishabh! Since you're a programming enthusiast in machine learning and deep learning and you're fond of Python, here are some project ideas tailored to your interests:

1. **House Price Prediction**: Use machine learning to predict house prices based on various features like location, size, and number of rooms. This could also tie into your civil engineering background.

2. **Image Classification**: Build a deep learning model using convolutional neural networks (CNNs) to classify images. You could use datasets related to construction or civil engineering.

3. **Natural Language Processing (NLP) Chatbot**: Create a chatbot that assists users in finding information related to civil engineering projects, materials, or construction guidelines.

4. **Smart Traffic Analysis**: Design a system that uses computer vision to analyz